In [1]:
# --- Работа с данными и СУБД ---
import numpy as np
import pandas as pd
from sqlalchemy import create_engine

# --- Машинное обучение и метрики ---
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import roc_auc_score

In [2]:
# функция для выгрузки данных 
engine = create_engine(
    "postgresql://robot-startml-ro:pheiph0hahj1Vaif@"
    "postgres.lab.karpov.courses:6432/startml"
)

def data_upload(q):
    with engine.connect() as conn:
        data = pd.read_sql(q, con=conn)
        return data

In [3]:
user_data = data_upload('SELECT * FROM "ajgerim-dubanaeva-mke5439_user_data"').drop(['index'], axis=1).set_index('user_id')
post_data = data_upload('SELECT * FROM "ajgerim-dubanaeva-mke5439_post_data"').drop(['index'], axis=1).set_index('post_id')

feed_data = pd.read_parquet('/kaggle/input/datasets/adubanaeva/feed-split/feed_eda.parquet')

In [4]:
user_data.dtypes

gender            int64
age               int64
country          object
city             object
exp_group         int64
os               object
source           object
user_avg_ctr    float64
dtype: object

In [5]:
# Преобразуем типы
user_data = user_data.astype({
    'country': 'category',
    'city': 'category',
    'exp_group': 'category',
    'os': 'category',
    'source': 'category'
})

In [6]:
post_data.dtypes

topic                    object
TextCluster              object
DistanceToCluster_0     float64
DistanceToCluster_1     float64
DistanceToCluster_2     float64
DistanceToCluster_3     float64
DistanceToCluster_4     float64
DistanceToCluster_5     float64
DistanceToCluster_6     float64
DistanceToCluster_7     float64
DistanceToCluster_8     float64
DistanceToCluster_9     float64
DistanceToCluster_10    float64
DistanceToCluster_11    float64
DistanceToCluster_12    float64
DistanceToCluster_13    float64
DistanceToCluster_14    float64
DistanceToCluster_15    float64
DistanceToCluster_16    float64
DistanceToCluster_17    float64
DistanceToCluster_18    float64
DistanceToCluster_19    float64
post_ctr                float64
dtype: object

In [7]:
# Преобразуем типы
post_data = post_data.astype({
    'topic': 'category',
    'TextCluster': 'category'
})

In [8]:
#Посчитаем показатели времени
feed_data['month'] = feed_data['timestamp'].dt.month.astype('category')
feed_data['hour'] = feed_data['timestamp'].dt.hour.astype('category')
feed_data['day_of_week'] = feed_data['timestamp'].dt.dayofweek.astype('category')
feed_data['is_weekend'] = (feed_data['timestamp'].dt.dayofweek >= 5).astype('int8').astype('category')

feed_data = feed_data.drop('timestamp', axis=1)

In [9]:
# собираем в один df
merged = feed_data.merge(user_data, on='user_id', how='left')
merged = merged.merge(post_data, on='post_id', how='left')

In [10]:
X_train = merged.loc[merged['set_type'] == 'train']
X_test = merged.loc[merged['set_type'] == 'test']

In [11]:
y_test = X_test['target']
X_test = X_test.drop(columns=['target', 'set_type'])

In [12]:
X_train.target.value_counts()

target
0    11312588
1     1641823
Name: count, dtype: int64

Сейчас в X_train слишком мало лайков, необходимо сбалансировать примерно 25%/75%

In [13]:
# 1. Отделяем лайки
pos_train = X_train[X_train['target'] == 1]

# 2. Берем в 3 раза больше обычных просмотров
# (Это даст нам пропорцию 25% лайков / 75% просто просмотров)
neg_train = X_train[X_train['target'] == 0].sample(n=len(pos_train) * 3, random_state=42)

# 3. Соединяем и перемешиваем (важно для градиентного бустинга)
X_train_balanced = pd.concat([pos_train, neg_train]).sample(frac=1, random_state=42)
y_train_balanced = X_train_balanced['target']

# 4. Удаляем таргет из признаков
X_train_balanced = X_train_balanced.drop(columns=['target', 'set_type'])

print(f"Размер сбалансированного трейна: {X_train_balanced.shape}")
print(f"Доля лайков: {y_train_balanced.mean():.2%}")

Размер сбалансированного трейна: (6567292, 37)
Доля лайков: 25.00%


In [14]:
print(X_train_balanced.isna().sum().sum())
print(X_test.isna().sum().sum())

0
0


In [15]:
X_train_balanced

,user_id,post_id,month,hour,day_of_week,is_weekend,gender,age,country,city,...,DistanceToCluster_11,DistanceToCluster_12,DistanceToCluster_13,DistanceToCluster_14,DistanceToCluster_15,DistanceToCluster_16,DistanceToCluster_17,DistanceToCluster_18,DistanceToCluster_19,post_ctr
7217524,160579,1834,12,19,6,1,1,22,Russia,Tyumen,...,0.517196,0.513603,0.424653,0.522707,0.432676,0.268183,0.458012,0.318436,0.620290,0.126344
9307530,68886,1836,12,8,3,0,0,20,Kazakhstan,Öskemen,...,0.525906,0.567580,0.442961,0.536120,0.439221,0.306319,0.478685,0.528931,0.625209,0.133524
6459776,165676,2488,12,13,5,1,1,19,Russia,Seversk,...,0.142312,0.564107,0.477622,0.556168,0.464676,0.433609,0.494397,0.554739,0.491108,0.105339
4586277,24076,1325,12,22,0,0,1,18,Russia,Blagodarnyy,...,0.471014,0.512532,0.344401,0.447342,0.391867,0.354874,0.360322,0.474298,0.579312,0.083913
12596340,49687,5097,12,17,0,0,0,16,Russia,Tolyatti,...,0.443082,0.476790,0.370525,0.481021,0.261075,0.356560,0.367973,0.452041,0.554643,0.163739
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11292641,78724,677,12,21,5,1,1,35,Russia,Chusovoy,...,0.505327,0.527749,0.407704,0.495696,0.420493,0.408825,0.378634,0.522353,0.605001,0.099039
6912986,102873,3726,12,10,6,1,1,15,Russia,Sharypovo,...,0.432378,0.677918,0.576275,0.657100,0.577147,0.578240,0.598420,0.668053,0.644936,0.128030
4265184,38945,1358,12,7,0,0,1,20,Russia,Saint Petersburg,...,0.461352,0.505109,0.333891,0.461860,0.409059,0.373770,0.386924,0.491694,0.571321,0.131689
8497547,60044,2499,12,21,1,0,1,21,Russia,Tyumen,...,0.531584,0.778287,0.699859,0.766365,0.705071,0.699595,0.709376,0.773866,0.644610,0.097938


In [16]:
#Список категориальных признаков
cat_features = [
    'user_id', 'post_id', 'month', 'hour', 'day_of_week', 'is_weekend',
    'gender', 'country', 'city', 'exp_group', 'os', 'source', 'topic', 'TextCluster'
]

#Создаем объекты Pool
train_pool = Pool(X_train_balanced, y_train_balanced, cat_features=cat_features)
test_pool = Pool(X_test, y_test, cat_features=cat_features)

In [17]:
model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.1,
    depth=6,   
    l2_leaf_reg=3,
    task_type="GPU",
    devices='0',
    loss_function='Logloss',
    eval_metric='AUC',
    metric_period=50,
    early_stopping_rounds=50,
    random_seed=42,
    verbose=50
)

model.fit(train_pool, eval_set=test_pool)

0:	test: 0.6503434	best: 0.6503434 (0)	total: 1.74s	remaining: 28m 53s
50:	test: 0.6673240	best: 0.6674205 (48)	total: 1m 14s	remaining: 23m 13s
100:	test: 0.6730099	best: 0.6730127 (99)	total: 2m 28s	remaining: 22m 4s
150:	test: 0.6729504	best: 0.6736659 (107)	total: 3m 45s	remaining: 21m 10s
bestTest = 0.6736658812
bestIteration = 107
Shrink model to first 108 iterations.


CatBoostClassifier(depth=6, devices='0', early_stopping_rounds=50, eval_metric='AUC', iterations=1000, l2_leaf_reg=3, learning_rate=0.1, loss_function='Logloss', metric_period=50, random_seed=42, task_type='GPU', verbose=50)

In [18]:
# Посмотрим важность признаков
feature_importance = model.get_feature_importance(train_pool)
feature_names = X_train_balanced.columns

fi_df = pd.DataFrame({
    'feature': feature_names,
    'importance': feature_importance
}).sort_values(by='importance', ascending=False)

# Топ-20
print("Топ-20 признаков:")
print(fi_df.head(20))

Топ-20 признаков:
                feature  importance
13         user_avg_ctr   35.540952
7                   age   27.318325
1               post_id   18.338701
0               user_id    8.964649
36             post_ctr    6.880355
9                  city    1.043545
3                  hour    0.788030
14                topic    0.579155
10            exp_group    0.288967
2                 month    0.230700
4           day_of_week    0.024388
8               country    0.002233
5            is_weekend    0.000000
12               source    0.000000
11                   os    0.000000
15          TextCluster    0.000000
16  DistanceToCluster_0    0.000000
17  DistanceToCluster_1    0.000000
18  DistanceToCluster_2    0.000000
19  DistanceToCluster_3    0.000000


In [19]:
# Базовый набор признаков (без мусора)
features_lite = ['user_avg_ctr', 'age', 'post_id', 'user_id', 'post_ctr', 'city', 
                 'hour', 'exp_group', 'topic', 'month']
cat_features = ['user_id', 'post_id', 'exp_group', 'city', 'hour', 'topic', 'month']


train_pool = Pool(X_train_balanced[features_lite], y_train_balanced, cat_features=cat_features)
test_pool = Pool(X_test[features_lite], y_test, cat_features=cat_features)

In [20]:
# Обучаем модель
model = CatBoostClassifier(iterations=1000, 
                           learning_rate=0.05, 
                           depth=6,   
                           l2_leaf_reg=2,
                           task_type="GPU", 
                           loss_function='Logloss', 
                           eval_metric='AUC', 
                           early_stopping_rounds=50,
                           random_seed=42, 
                           verbose=50, 
                           metric_period=50)
model.fit(train_pool, eval_set=test_pool)

0:	test: 0.6503434	best: 0.6503434 (0)	total: 727ms	remaining: 12m 5s
50:	test: 0.6614256	best: 0.6624061 (36)	total: 36.6s	remaining: 11m 21s
100:	test: 0.6667084	best: 0.6667084 (100)	total: 1m 14s	remaining: 11m 7s
150:	test: 0.6707493	best: 0.6707493 (150)	total: 1m 55s	remaining: 10m 49s
200:	test: 0.6731540	best: 0.6733263 (190)	total: 2m 35s	remaining: 10m 16s
250:	test: 0.6737435	best: 0.6737619 (248)	total: 3m 15s	remaining: 9m 44s
300:	test: 0.6741824	best: 0.6743153 (291)	total: 3m 57s	remaining: 9m 11s
bestTest = 0.6743152738
bestIteration = 291
Shrink model to first 292 iterations.


CatBoostClassifier(depth=6, early_stopping_rounds=50, eval_metric='AUC', iterations=1000, l2_leaf_reg=2, learning_rate=0.05, loss_function='Logloss', metric_period=50, random_seed=42, task_type='GPU', verbose=50)

In [21]:
model.save_model('catboost_model.cbm', format="cbm")

In [22]:
user_data_fin = user_data[['age', 'city', 'exp_group', 'user_avg_ctr']]

In [23]:
post_data_fin = post_data[['topic', 'post_ctr']]

In [25]:
user_data_fin.to_sql('ajgerim-dubanaeva-mke5439_user_data_fin', con=engine, if_exists='replace')

205

In [26]:
post_data_fin.to_sql('ajgerim-dubanaeva-mke5439_post_data_fin', con=engine, if_exists='replace')

23